# Imports

In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from torchinfo import summary
from tqdm import tqdm
from sklearn.metrics import r2_score

# Constants

In [2]:
device = 'cuda'

# Dataset

In [3]:
class TupleDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.from_numpy(x)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]
    
    def shape(self):
        return self.x.shape

In [4]:
train_dataset = torch.load('../model_creation/train_dataset.pt')
test_dataset = torch.load('../model_creation/test_dataset.pt')

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

C:\Users\agile\AppData\Local\Temp\ipykernel_39580\3963001541.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_dataset = torch.load('../model_creation/train_dataset.

# Models

In [5]:
class BiGRU(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        
        self.bigru = nn.GRU(
            input_features, 64,
            num_layers=3,
            bidirectional=True,
            batch_first=True,
            dropout=0.2
        )
        
        self.fc = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        _, h = self.bigru(x)
        
        # h shape: (num_layers * num_directions, batch, hidden)
        num_layers = self.bigru.num_layers
        num_directions = 2 if self.bigru.bidirectional else 1
        
        h = h.view(num_layers, num_directions, x.size(0), 64)
        h = h[-1]  # last layer → (2, batch, 64)
        
        h = torch.cat((h[0], h[1]), dim=1)  # (batch, 128)
        
        return self.fc(h)

In [6]:
class BiLSTM(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        
        self.bilstm = nn.LSTM(
            input_features, 64,
            num_layers=3,
            bidirectional=True,
            batch_first=True,
            dropout=0.2
        )
        
        self.fc = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        _, (h, _) = self.bilstm(x)
        
        # shape: (num_layers * num_directions, batch, hidden)
        h = h.view(3, 2, x.size(0), 64)  # (layers, directions, batch, hidden)
        h = h[-1]                        # last layer -> (2, batch, 64)
        
        h = torch.cat((h[0], h[1]), dim=1)  # (batch, 128)
        
        return self.fc(h)

In [7]:
class FC(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        
        self.fc1 = nn.Sequential(
            nn.Linear(input_features, 16),
            nn.ReLU(),

            nn.Linear(16, 32),
            nn.ReLU(),

            nn.Linear(32, 128)
        )
        
        self.fc = nn.Sequential(
            nn.Linear(128, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        x = x.squeeze(-1)
        out = self.fc1(x)
        
        return self.fc(out)

In [8]:
bigru_model = BiGRU(input_features=1)
bigru_model.load_state_dict(torch.load('../model_creation/bigru_model.pth'))
bigru_model.eval()
bigru_model.fc = nn.Identity()

bilstm_model = BiLSTM(input_features=1)
bilstm_model.load_state_dict(torch.load('../model_creation/bilstm_model.pth'))
bilstm_model.eval()
bilstm_model.fc = nn.Identity()

fc_model = FC(input_features=99)
fc_model.load_state_dict(torch.load('../model_creation/fc_model.pth'))
fc_model.eval()
fc_model.fc = nn.Identity()

bigru_model.to(device)
bilstm_model.to(device)
fc_model.to(device)

C:\Users\agile\AppData\Local\Temp\ipykernel_39580\3343140410.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  bigru_model.load_state_dict(torch.load('../model_creation/bi

FC(
  (fc1): Sequential(
    (0): Linear(in_features=99, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=128, bias=True)
  )
  (fc): Identity()
)

# Merge

In [9]:
def merge_outputs(loader):
    merged_x = []
    merged_y = []

    with torch.no_grad():
        for inputs, targets in loader:
            bigru_model_outputs = bigru_model(inputs.to(device))
            bilstm_model_outputs = bilstm_model(inputs.to(device))
            fc_model_outputs = fc_model(inputs.to(device))
            concated_outputs = torch.concat([bigru_model_outputs, bilstm_model_outputs, fc_model_outputs], dim=-1)
            
            merged_x.append(concated_outputs)
            merged_y.append(targets)

    merged_x_pt = torch.cat(merged_x)
    merged_y_pt = torch.cat(merged_y)
    return merged_x_pt.cpu().numpy(), merged_y_pt.cpu().numpy()

In [10]:
train_merged_x, train_merged_y = merge_outputs(train_loader)
train_merged_y = train_merged_y.flatten()
test_merged_x, test_merged_y = merge_outputs(test_loader)
test_merged_y = test_merged_y.flatten()

In [11]:
train_df = pd.DataFrame(train_merged_x)
train_df["target"] = train_merged_y

test_df = pd.DataFrame(test_merged_x)
test_df["target"] = test_merged_y

In [12]:
print(len(train_df))
print(len(test_df))

7636
3216


# Save Merged Data

In [13]:
train_df.to_csv('merged_train.csv')
test_df.to_csv('merged_test.csv')